In [ ]:
## Day 2 Banking Assignment
Create/use a banking transactions DataFrame with: `transaction_id, customer_id, account_type, transaction_type, amount, city, transaction_date`.

Tasks:
1. Select transaction ID, customer ID and amount.
2. Filter transactions above ₹50,000.
3. Filter only `DEBIT` transactions.
4. Create `transaction_fee` as 1% of amount.
5. Create `net_amount` after fee.
6. Rename `city` to `branch_city`.
7. Drop an unnecessary column.
8. Find distinct account types.
9. Remove duplicate transaction IDs.
10. Sort transactions by amount descending.
11. Group by account type and calculate count, total amount, average amount, min and max.
12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.
13. Convert transaction date to DateType and extract year/month/day.
14. Handle null amount/city values appropriately.
15. Create another DataFrame and perform `unionByName`.
16. Create a customer master DataFrame and join it with transactions.
17. Repartition by account type and explain why.
18. Build a final banking pipeline producing clean, enriched transactions.



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType
)
import datetime
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BankingTransactions").master("local[*]").getOrCreate()


schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("account_type", StringType(), False),
    StructField("transaction_type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("city", StringType(), False),
    StructField("transaction_date", DateType(), False),
])

data = [
    (1,  501, "SAVINGS",  "DEPOSIT",    1500.00, "Bengaluru", datetime.date(2024, 1, 5)),
    (2,  502, "CURRENT",  "WITHDRAWAL", 300.00,  "Mumbai",    datetime.date(2024, 1, 7)),
    (3,  501, "SAVINGS",  "WITHDRAWAL", 200.00,  "Bengaluru", datetime.date(2024, 1, 10)),
    (4,  503, "SAVINGS",  "DEPOSIT",    5000.00, "Delhi",     datetime.date(2024, 1, 12)),
    (5,  504, "CURRENT",  "TRANSFER",   750.00,  "Chennai",   datetime.date(2024, 1, 15)),
    (6,  502, "CURRENT",  "DEPOSIT",    1200.00, "Mumbai",    datetime.date(2024, 1, 18)),
    (7,  505, "SAVINGS",  "WITHDRAWAL", 400.00,  "Hyderabad", datetime.date(2024, 1, 20)),
    (8,  503, "SAVINGS",  "TRANSFER",   1000.00, "Delhi",     datetime.date(2024, 1, 22)),
    (9,  506, "CURRENT",  "DEPOSIT",    2500.00, "Pune",      datetime.date(2024, 2, 1)),
    (10, 501, "SAVINGS",  "DEPOSIT",    600.00,  "Bengaluru", datetime.date(2024, 2, 3)),
    (11, 504, "CURRENT",  "WITHDRAWAL", 150.00,  "Chennai",   datetime.date(2024, 2, 5)),
    (12, 507, "SAVINGS",  "DEPOSIT",    3200.00, "Kolkata",   datetime.date(2024, 2, 8)),
    (13, 505, "SAVINGS",  "TRANSFER",   900.00,  "Hyderabad", datetime.date(2024, 2, 11)),
    (14, 506, "CURRENT",  "WITHDRAWAL", 450.00,  "Pune",      datetime.date(2024, 2, 14)),
    (15, 507, "SAVINGS",  "DEPOSIT",    1800.00, "Kolkata",   datetime.date(2024, 2, 17)),
    (16, 502, "CURRENT",  "TRANSFER",   275.50,  "Mumbai",    datetime.date(2024, 2, 20)),
]

df_bank = spark.createDataFrame(data, schema)
df_bank.show()


+--------------+-----------+------------+----------------+------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|
+--------------+-----------+------------+----------------+------+---------+----------------+
|             1|        501|     SAVINGS|         DEPOSIT|1500.0|Bengaluru|      2024-01-05|
|             2|        502|     CURRENT|      WITHDRAWAL| 300.0|   Mumbai|      2024-01-07|
|             3|        501|     SAVINGS|      WITHDRAWAL| 200.0|Bengaluru|      2024-01-10|
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|    Delhi|      2024-01-12|
|             5|        504|     CURRENT|        TRANSFER| 750.0|  Chennai|      2024-01-15|
|             6|        502|     CURRENT|         DEPOSIT|1200.0|   Mumbai|      2024-01-18|
|             7|        505|     SAVINGS|      WITHDRAWAL| 400.0|Hyderabad|      2024-01-20|
|             8|        503|     SAVINGS|        TRANSFER|1000.0|    D

In [7]:
df_bank.select("transaction_id","customer_id","amount").show()

+--------------+-----------+------+
|transaction_id|customer_id|amount|
+--------------+-----------+------+
|             1|        501|1500.0|
|             2|        502| 300.0|
|             3|        501| 200.0|
|             4|        503|5000.0|
|             5|        504| 750.0|
|             6|        502|1200.0|
|             7|        505| 400.0|
|             8|        503|1000.0|
|             9|        506|2500.0|
|            10|        501| 600.0|
|            11|        504| 150.0|
|            12|        507|3200.0|
|            13|        505| 900.0|
|            14|        506| 450.0|
|            15|        507|1800.0|
|            16|        502| 275.5|
+--------------+-----------+------+



In [9]:
df_bank.filter(df_bank.amount > 1000).show()


+--------------+-----------+------------+----------------+------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|
+--------------+-----------+------------+----------------+------+---------+----------------+
|             1|        501|     SAVINGS|         DEPOSIT|1500.0|Bengaluru|      2024-01-05|
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|    Delhi|      2024-01-12|
|             6|        502|     CURRENT|         DEPOSIT|1200.0|   Mumbai|      2024-01-18|
|             9|        506|     CURRENT|         DEPOSIT|2500.0|     Pune|      2024-02-01|
|            12|        507|     SAVINGS|         DEPOSIT|3200.0|  Kolkata|      2024-02-08|
|            15|        507|     SAVINGS|         DEPOSIT|1800.0|  Kolkata|      2024-02-17|
+--------------+-----------+------------+----------------+------+---------+----------------+



In [11]:
df_bank.filter(df_bank.transaction_type=="WITHDRAWAL").show()

+--------------+-----------+------------+----------------+------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|
+--------------+-----------+------------+----------------+------+---------+----------------+
|             2|        502|     CURRENT|      WITHDRAWAL| 300.0|   Mumbai|      2024-01-07|
|             3|        501|     SAVINGS|      WITHDRAWAL| 200.0|Bengaluru|      2024-01-10|
|             7|        505|     SAVINGS|      WITHDRAWAL| 400.0|Hyderabad|      2024-01-20|
|            11|        504|     CURRENT|      WITHDRAWAL| 150.0|  Chennai|      2024-02-05|
|            14|        506|     CURRENT|      WITHDRAWAL| 450.0|     Pune|      2024-02-14|
+--------------+-----------+------------+----------------+------+---------+----------------+



In [4]:
from pyspark.sql.functions import col, round as spark_round
df_bank = df_bank.withColumn("transaction_fee",spark_round(col("amount")*0.01,2))
df_bank = df_bank.withColumn("net_amount",col("amount")-col("transaction_fee"))
#df_bank = df_bank.withColumn("branch_city",col("city"))
#df_bank = df_bank.drop("city")
df_bank.show()




+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|transaction_fee|net_amount|
+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+
|             1|        501|     SAVINGS|         DEPOSIT|1500.0|Bengaluru|      2024-01-05|           15.0|    1485.0|
|             2|        502|     CURRENT|      WITHDRAWAL| 300.0|   Mumbai|      2024-01-07|            3.0|     297.0|
|             3|        501|     SAVINGS|      WITHDRAWAL| 200.0|Bengaluru|      2024-01-10|            2.0|     198.0|
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|    Delhi|      2024-01-12|           50.0|    4950.0|
|             5|        504|     CURRENT|        TRANSFER| 750.0|  Chennai|      2024-01-15|            7.5|     742.5|
|             6|        502|     CURRENT

In [34]:
df_bank.select("account_type").distinct().show()

+------------+
|account_type|
+------------+
|     SAVINGS|
|     CURRENT|
+------------+



In [38]:
df_bank = df_bank.dropDuplicates(["transaction_id"])
df_bank.show()

+--------------+-----------+------------+----------------+------+----------------+---------------+----------+-----------+
|transaction_id|customer_id|account_type|transaction_type|amount|transaction_date|transaction_fee|net_amount|branch_city|
+--------------+-----------+------------+----------------+------+----------------+---------------+----------+-----------+
|             1|        501|     SAVINGS|         DEPOSIT|1500.0|      2024-01-05|           15.0|    1485.0|  Bengaluru|
|             2|        502|     CURRENT|      WITHDRAWAL| 300.0|      2024-01-07|            3.0|     297.0|     Mumbai|
|             3|        501|     SAVINGS|      WITHDRAWAL| 200.0|      2024-01-10|            2.0|     198.0|  Bengaluru|
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|      2024-01-12|           50.0|    4950.0|      Delhi|
|             5|        504|     CURRENT|        TRANSFER| 750.0|      2024-01-15|            7.5|     742.5|    Chennai|
|             6|        

In [6]:
df_bank = df_bank.orderBy(col("amount").desc())
df_bank.show()

+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|transaction_fee|net_amount|
+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|    Delhi|      2024-01-12|           50.0|    4950.0|
|            12|        507|     SAVINGS|         DEPOSIT|3200.0|  Kolkata|      2024-02-08|           32.0|    3168.0|
|             9|        506|     CURRENT|         DEPOSIT|2500.0|     Pune|      2024-02-01|           25.0|    2475.0|
|            15|        507|     SAVINGS|         DEPOSIT|1800.0|  Kolkata|      2024-02-17|           18.0|    1782.0|
|             1|        501|     SAVINGS|         DEPOSIT|1500.0|Bengaluru|      2024-01-05|           15.0|    1485.0|
|             6|        502|     CURRENT

In [9]:
#Group by account type and calculate count, total amount, average amount, min and max.

df_bank.groupBy("account_type").agg(F.count('*').alias("transaction_amount"),
                                    F.sum("amount").alias("total_amount"),
                                    F.avg("amount").alias("avg_amount"),
                                    F.min("amount").alias("min_amount"),
                                    F.max("amount").alias("max_amount")
).show()



+------------+------------------+------------+------------------+----------+----------+
|account_type|transaction_amount|total_amount|        avg_amount|min_amount|max_amount|
+------------+------------------+------------+------------------+----------+----------+
|     SAVINGS|                 9|     14600.0|1622.2222222222222|     200.0|    5000.0|
|     CURRENT|                 7|      5625.5| 803.6428571428571|     150.0|    2500.0|
+------------+------------------+------------+------------------+----------+----------+



In [18]:
# Convert transaction date to DateType and extract year/month/day.
#df_bank.printSchema()
from pyspark.sql.functions import to_date, year, month, dayofmonth, col
df_bank = (df_bank.withColumn("txn_year",year(col("transaction_date")))
                  .withColumn("txn_month",month(col("transaction_date")))
                  .withColumn("txn_day",dayofmonth(col("transaction_date")))
          ).show()

+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+--------+---------+-------+
|transaction_id|customer_id|account_type|transaction_type|amount|     city|transaction_date|transaction_fee|net_amount|txn_year|txn_month|txn_day|
+--------------+-----------+------------+----------------+------+---------+----------------+---------------+----------+--------+---------+-------+
|             4|        503|     SAVINGS|         DEPOSIT|5000.0|    Delhi|      2024-01-12|           50.0|    4950.0|    2024|        1|     12|
|            12|        507|     SAVINGS|         DEPOSIT|3200.0|  Kolkata|      2024-02-08|           32.0|    3168.0|    2024|        2|      8|
|             9|        506|     CURRENT|         DEPOSIT|2500.0|     Pune|      2024-02-01|           25.0|    2475.0|    2024|        2|      1|
|            15|        507|     SAVINGS|         DEPOSIT|1800.0|  Kolkata|      2024-02-17|           18.0|    1782.0